### Connect to SQLite

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt

conn = sqlite3.connect("../coffee_analysis.db")

### Inspect the tables

In [ ]:
# Show table names
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)

In [ ]:
# Preview each table
pd.read_sql_query("SELECT * FROM business LIMIT 5;", conn)
pd.read_sql_query("SELECT * FROM reviews LIMIT 5;", conn)
pd.read_sql_query("SELECT * FROM tips LIMIT 5;", conn)
pd.read_sql_query("SELECT * FROM checkins LIMIT 5;", conn)
pd.read_sql_query("SELECT * FROM users LIMIT 5;", conn)

In [ ]:
# Inspect the tables
pd.read_sql_query("PRAGMA table_info(business);", conn)
pd.read_sql_query("PRAGMA table_info(reviews);", conn)
pd.read_sql_query("PRAGMA table_info(tips);", conn)
pd.read_sql_query("PRAGMA table_info(checkins);", conn)
pd.read_sql_query("PRAGMA table_info(users);", conn)

### ERD
```mermaid
erDiagram
    BUSINESS ||--o{ REVIEWS : has
    BUSINESS ||--o{ TIPS : has
    BUSINESS ||--o{ CHECKINS : has
    USERS ||--o{ REVIEWS : writes
    USERS ||--o{ TIPS : writes

    BUSINESS {
        string business_id PK
        string name
        string city
        string state
        float latitude
        float longitude
        float stars
        int review_count
        string categories
        int is_open
    }

    REVIEWS {
        string review_id PK
        string user_id FK
        string business_id FK
        int stars
        int useful
        int funny
        int cool
        string text
        datetime date
        int review_length
    }

    TIPS {
        string user_id FK
        string business_id FK
        string text
        datetime date
        int compliment_count
        int tip_length
    }

    CHECKINS {
        string business_id FK
        string date
        string checkin_dates
        int total_checkins
    }

    USERS {
        string user_id PK
        int review_count
        datetime yelping_since
        int useful
        int funny
        int cool
        int fans
        float average_stars
    }
```

### Top cities by number of coffee shops

In [ ]:
query = """
SELECT city, COUNT(*) AS total_coffee_shops
FROM business
GROUP BY city
ORDER BY total_coffee_shops DESC
LIMIT 10;
"""
top_cities = pd.read_sql_query(query, conn)
top_cities

### Top businesses by review count

In [ ]:
query = """
SELECT name, city, stars, review_count
FROM business
ORDER BY review_count DESC
LIMIT 10;
"""
top_businesses = pd.read_sql_query(query, conn)
top_businesses

### City summary

In [ ]:
query = """
SELECT city,
       COUNT(*) AS total_shops,
       ROUND(AVG(stars), 2) AS avg_rating,
       ROUND(AVG(review_count), 2) AS avg_reviews
FROM business
GROUP BY city
HAVING COUNT(*) >= 3
ORDER BY total_shops DESC;
"""
city_summary = pd.read_sql_query(query, conn)
city_summary

### Average rating by city

In [ ]:
city_summary.head(10).plot(kind="bar", x="city", y="avg_rating", legend=False)
plt.title("Average Rating by City")
plt.xlabel("City")
plt.ylabel("Average Rating")
plt.tight_layout()
plt.show()

### Top cities chart

In [ ]:
top_cities.plot(kind="bar", x="city", y="total_coffee_shops", legend=False)
plt.title("Top Cities by Number of Coffee Shops")
plt.xlabel("City")
plt.ylabel("Number of Coffee Shops")
plt.tight_layout()
plt.show()

### Business engagement

In [ ]:
query = """
SELECT b.name,
       b.city,
       b.stars,
       b.review_count,
       COALESCE(c.total_checkins, 0) AS total_checkins
FROM business b
LEFT JOIN checkins c
  ON b.business_id = c.business_id
ORDER BY total_checkins DESC
LIMIT 10;
"""
top_checkins = pd.read_sql_query(query, conn)
top_checkins

### Review count vs rating

In [ ]:
business_df = pd.read_sql_query("SELECT review_count, stars FROM business;", conn)

plt.scatter(business_df["review_count"], business_df["stars"])
plt.title("Review Count vs Business Rating")
plt.xlabel("Review Count")
plt.ylabel("Stars")
plt.tight_layout()
plt.show()

### Average review length by business

In [ ]:
query = """
SELECT b.name,
       b.city,
       ROUND(AVG(r.review_length), 2) AS avg_review_length,
       COUNT(r.review_id) AS total_reviews
FROM reviews r
JOIN business b
  ON r.business_id = b.business_id
GROUP BY b.business_id, b.name, b.city
HAVING COUNT(r.review_id) >= 10
ORDER BY avg_review_length DESC
LIMIT 10;
"""
review_length_summary = pd.read_sql_query(query, conn)
review_length_summary